In [5]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

# YouGov perceptions (9–14 Nov 2025) vs reality (IFS). rank 1 = most spent.
perc = ["Debt interest","NHS","Working age benefits","Public order & safety","Defence",
        "Pensioners","Education","Social care","Overseas aid","Housing","Transport"]
real = ["NHS","Working age benefits","Pensioners","Education","Debt interest","Defence",
        "Public order & safety","Transport","Social care","Housing","Overseas aid"]

rows = []
for cat in perc:
    rows.append((cat, "Public thinks", perc.index(cat)+1, 0))
    rows.append((cat, "Reality",       real.index(cat)+1, 1))
df = pd.DataFrame(rows, columns=["cat","side","rank","xpos"])
df["hl"] = df["cat"] == "Debt interest"

RED, GREY, INKGREY = "#e6224b", "#cdd2dc", "#4a4f60"
base = alt.Chart(df)
ex = alt.X("xpos:Q", scale=alt.Scale(domain=[-0.05, 1.05]), axis=None)
ey = alt.Y("rank:Q", scale=alt.Scale(reverse=True, domain=[0.3, 11.7]), axis=None)

lines = base.mark_line().encode(x=ex, y=ey, detail="cat:N",
    color=alt.condition("datum.hl", alt.value(RED), alt.value(GREY)),
    size=alt.condition("datum.hl", alt.value(3.5), alt.value(1.4)),
    opacity=alt.condition("datum.hl", alt.value(1), alt.value(0.45)))
pts = base.mark_circle(size=60).encode(x=ex, y=ey,
    color=alt.condition("datum.hl", alt.value(RED), alt.value(GREY)),
    opacity=alt.condition("datum.hl", alt.value(1), alt.value(0.45)))

def txt(side, align, dx, hl, bold):
    d = df[(df.side == side) & (df.hl == hl)]
    return alt.Chart(d).mark_text(align=align, dx=dx, fontSize=12,
        fontWeight="bold" if bold else "normal",
        color=RED if hl else INKGREY).encode(x=ex, y=ey, text="cat:N")

labels = (txt("Public thinks","right",-12,False,False) + txt("Public thinks","right",-12,True,True)
        + txt("Reality","left",12,False,False)          + txt("Reality","left",12,True,True))


chart = (lines + pts + labels).properties(
    width=300, height=470, padding={"left":150, "right":130, "top":10, "bottom":10},
    title=alt.Title("Debt interest: what the public thinks vs reality",
        subtitle="Perceived vs actual government spending, ranked 1–11"))

styles.save(chart, name="perception_vs_reality", svg=True)
chart

alt.LayerChart(...)